# 4 - Summary & plots

**Stage 4 of 4.** Loads the portfolio return series from notebook 3 and produces the reporting layer:

- Final performance table across all construction levels + Sharpe-by-year
- Three-panel equity-curve figure (levels, gross vs net, drawdown) -> `Results/equity_curves.png`
- Paper-trading signal generation for the latest date (retrains the pre-registered ElasticNet)
- The **headline net Sharpe** printed at the very end


In [ ]:
# =============================================================
# Cell 33b - Factor exposure of the traded signal (raw vs neutralized)
# =============================================================
# Large raw loadings on momentum / volatility mean the signal is
# mostly factor beta; the neutralized bars collapse to ~0.
import json
_expo_path = "Data/interim/factor_exposures.json"
if os.path.exists(_expo_path):
    _ex = json.load(open(_expo_path))
    _facs = list(_ex["before"].keys())
    _b = [_ex["before"][f] for f in _facs]; _a = [_ex["after"][f] for f in _facs]
    fig, ax = plt.subplots(figsize=(11, 5))
    x = np.arange(len(_facs)); wd = 0.38
    ax.bar(x - wd / 2, _b, wd, label="raw signal", color="steelblue")
    ax.bar(x + wd / 2, _a, wd, label="neutralized", color="darkorange")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels(_facs, rotation=25, ha="right")
    ax.set_ylabel("mean per-date Spearman corr")
    ax.set_title("Traded-signal factor exposure — raw vs neutralized\n"
                 "(large raw loadings on momentum & volatility = the edge is mostly factor beta)")
    ax.legend(); ax.grid(True, axis="y", alpha=0.3); plt.tight_layout()
    plt.savefig("Results/factor_exposure.png", dpi=150, bbox_inches="tight"); plt.show()
    print("Saved -> Results/factor_exposure.png")
else:
    print(f"{_expo_path} not found — run portfolio_sizing.ipynb (Cell 27b) first.")


In [ ]:
# =============================================================
# Cell 33c - Net equity curves: raw L5 vs neutralized vs Level 6
# =============================================================
fig, ax = plt.subplots(figsize=(13, 6))
_series = [("Level 5 net (+sqrt-impact @100M)", daily_net_full, "steelblue"),
           ("Level 6 net (sqrt-impact in objective)", daily_l6_net, "seagreen")]
if daily_neut_net is not None:
    _series.append(("Neutralized L5 net (factor-neutral)", daily_neut_net, "crimson"))
for _lbl, _ser, _col in _series:
    _eq = (1 + _ser.dropna()).cumprod()
    ax.plot(_eq.index, _eq.values, label=_lbl, color=_col, lw=1.7)
ax.axhline(1.0, color="black", ls="--", lw=0.8)
ax.set_title("Net-of-cost equity curves — raw vs factor-neutral vs cost-aware (Level 6)")
ax.set_ylabel("Cumulative Return"); ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig("Results/net_curves_compare.png", dpi=150, bbox_inches="tight"); plt.show()
print("Saved -> Results/net_curves_compare.png")


In [ ]:
# --- Imports & setup -----------------------------------------
# Resolve paths from the repo root no matter where Jupyter launched.
import os
if os.path.basename(os.getcwd()) == "Notebooks":
    os.chdir("..")
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cvxpy as cp
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.6f}".format)


In [ ]:
# --- Load portfolio returns, model_df, metadata & params -----
port_returns = pd.read_parquet("Data/interim/portfolio_returns.parquet")
daily_l2        = port_returns["l2"].dropna()
daily_l4        = port_returns["l4"].dropna()
daily_l5        = port_returns["l5"].dropna()
daily_net_10bps = port_returns["net_10bps"].dropna()
daily_net_full  = (port_returns["net_full"].dropna()
                   if "net_full" in port_returns.columns else daily_net_10bps)
daily_l6_net    = (port_returns["l6_net"].dropna()
                   if "l6_net" in port_returns.columns else daily_net_full)
daily_neut_net  = (port_returns["neut_net"].dropna()
                   if "neut_net" in port_returns.columns else None)

model_df = pd.read_parquet("Data/interim/model_df.parquet")
model_df["date"] = pd.to_datetime(model_df["date"])

with open("Data/interim/pipeline_meta.json") as f:
    meta = json.load(f)
with open("Data/interim/best_params.json") as f:
    best_params = json.load(f)

ALL_FEATURES     = meta["ALL_FEATURES"]
TARGET_HORIZON   = meta["TARGET_HORIZON"]
PERIODS_PER_YEAR = meta["PERIODS_PER_YEAR"]
best_en_params   = tuple(best_params["best_en_params"])

# Constants reused by the paper-trading optimiser (from Level 5, notebook 3)
RISK_AVERSION = 1.0
MAX_WEIGHT    = 0.02

def sharpe(daily_returns, ann_factor=252):
    """Annualised Sharpe ratio from a per-period return series."""
    mu, sig = daily_returns.mean(), daily_returns.std()
    return 0.0 if sig < 1e-10 else mu / sig * np.sqrt(ann_factor)

# Recompute the net-of-cost equity/drawdown the headline block prints
sharpe_net = sharpe(daily_net_10bps, PERIODS_PER_YEAR)
equity_net = (1 + daily_net_10bps).cumprod()
max_dd_net = (equity_net / equity_net.cummax() - 1).min()

print(f"Loaded portfolio returns  |  headline net Sharpe = {sharpe_net:.4f}")


In [ ]:
# =============================================================
# Cell 32 — Final Performance Summary
# =============================================================
# Side-by-side comparison across all portfolio construction levels.
# This is the answer to the assignment question:
# "What is your final ROI/Sharpe?"
#
# We report Level 5 net-of-cost as our headline result.
#
# NOTE ON SHARPE:
# From course material: "Few PMs have Sharpe > 1. High Sharpes
# are not believable." If gross Sharpe is much above 1, we flag
# the survivorship bias fix in v11 and the limited sample period
# as potential explanations. Net Sharpe after costs is more
# representative of real-world performance.
# =============================================================

print("=" * 70)
print("Cell 32: FINAL PERFORMANCE SUMMARY")
print("=" * 70)

summary_data = {
    "Level 2 (Gross Notional)": daily_l2,
    "Level 4 (Vol Targeted)":   daily_l4,
    "Level 5 MV Gross":         daily_l5,
    "Level 5 MV Net (liq-adj 10bps)": daily_net_10bps,
    "Level 5 MV Net (+sqrt-impact $100M)": daily_net_full,
    "Level 6 MV Net (sqrt-in-obj $100M)": daily_l6_net,
}

print(f"\n{'Portfolio':<32} {'Sharpe':>8} {'Ann.Vol':>8} {'MaxDD':>8}"
      f" {'TotalRet':>10} {'WinRate':>8}")
print("-" * 78)

for label, series in summary_data.items():
    series = series.dropna()
    s  = sharpe(series, PERIODS_PER_YEAR)   # v19: period series
    v  = series.std() * np.sqrt(PERIODS_PER_YEAR)   # v19
    eq = (1 + series).cumprod()
    dd = (eq / eq.cummax() - 1).min()
    tr = eq.iloc[-1] - 1
    wr = (series > 0).mean()
    print(f"  {label:<30} {s:>8.3f} {v:>8.4f} {dd:>8.4f} {tr:>10.4f} {wr:>8.1%}")

# ── v17 (change #2): Sharpe by calendar year ──
print("\n" + "=" * 70)
print("Cell 32: SHARPE BY YEAR")
print("=" * 70)
_years = sorted({d.year for s in summary_data.values() for d in s.dropna().index})
_hdr = f"  {'Portfolio':<32}" + "".join(f"{y:>9}" for y in _years)
print(_hdr)
print("  " + "-" * (len(_hdr) - 2))
for label, series in summary_data.items():
    s = series.dropna()
    by_year = s.groupby(s.index.year).apply(
        lambda x: sharpe(x, PERIODS_PER_YEAR) if len(x) >= 20 else np.nan
    )
    row = f"  {label:<32}"
    for y in _years:
        v = by_year.get(y, np.nan)
        row += f"{v:>9.2f}" if np.isfinite(v) else f"{'—':>9}"
    print(row)
print("\n  Note: years with < 20 rebalance periods in-sample are shown as '—'")
print("  (annualising a Sharpe from a short stub period is unreliable).")

print("\n" + "=" * 70)
print("Cell 32: HEADLINE RESULT — Level 5 MV (net of liquidity-adjusted costs,")
print("Cell 32:                   10 bps for the median-liquidity name)")
print(f"  Sharpe (annualised): {sharpe_net:.4f}")
print(f"  Max drawdown:        {max_dd_net:.4f}")
print(f"  Total return:        {equity_net.iloc[-1]-1:.4f}")
print("=" * 70)
print("\nCell 32: Limitations:")
print("  - Point-in-time constituents used (survivorship bias fixed vs v10)")
print("  - Cost model: linear sigma/sqrt(ADV)-scaled, 10bps median name")
print("    (excludes borrow cost and nonlinear square-root impact)")
print("  - Short interest available Dec 2022+ only (NaN → dropped before that)")
print("  - FRED macro lagged 1 day (full rigor requires vintage release dates)")

In [ ]:
# =============================================================
# Cell 33 — Equity Curve Visualisation
# =============================================================
# Three-panel chart:
# 1. Portfolio construction level comparison (gross)
# 2. Level 5 gross vs net of costs
# 3. Level 5 drawdown profile (net)
# =============================================================

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle("Portfolio Equity Curves\n"
             "(PIT universe · demeaned 5-day cross-sectional target · pre-registered walk-forward ElasticNet)",
             fontsize=13, fontweight="bold")

ax1 = axes[0]
ax1.set_title("Portfolio Construction Level Comparison (Gross)")
for label, series in [
    ("Level 2 — Gross Notional", daily_l2),
    ("Level 4 — Vol Targeted",   daily_l4),
    ("Level 5 — Mean-Variance",  daily_l5),
]:
    eq = (1 + series.dropna()).cumprod()
    ax1.plot(eq.index, eq.values, label=label, linewidth=1.5)
ax1.axhline(1.0, color="black", linestyle="--", linewidth=0.8)
ax1.legend(loc="upper left", fontsize=9)
ax1.set_ylabel("Cumulative Return")
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.set_title("Level 5: Gross vs. Net of Transaction Costs (10 bps)")
eq_gross = (1 + daily_l5.dropna()).cumprod()
eq_net   = (1 + daily_net_10bps.dropna()).cumprod()
ax2.plot(eq_gross.index, eq_gross.values, label="Gross", linewidth=1.5, color="steelblue")
ax2.plot(eq_net.index,   eq_net.values,   label="Net (10bps)", linewidth=1.5, color="darkorange")
ax2.axhline(1.0, color="black", linestyle="--", linewidth=0.8)
ax2.legend(loc="upper left", fontsize=9)
ax2.set_ylabel("Cumulative Return")
ax2.grid(True, alpha=0.3)

ax3 = axes[2]
ax3.set_title("Level 5 Drawdown (Net of Costs)")
eq_net_dd = eq_net / eq_net.cummax() - 1
ax3.fill_between(eq_net_dd.index, eq_net_dd.values, 0,
                 alpha=0.5, color="red", label="Drawdown")
ax3.axhline(0, color="black", linewidth=0.8)
ax3.legend(loc="lower left", fontsize=9)
ax3.set_ylabel("Drawdown")
ax3.grid(True, alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

plt.tight_layout()
plt.savefig("Results/equity_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Cell 33: Chart saved to Results/equity_curves.png")

In [ ]:
# =============================================================
# Cell 34 — Paper Trading — Signal Generation
# =============================================================
# Generates live target positions for today using the v18
# pipeline: point-in-time universe, residualized cross-sectional
# features, the PRE-REGISTERED ElasticNet retrained on the last
# 2 years (predicting the demeaned 5-day forward return), and
# Level 5 CVXPY construction.
#
# IN PRODUCTION: run this cell daily on a scheduler.
# Output feeds into a broker API or EMS.
# Fill assumption: execution at next-day close price.
#
# SOLVER: SCS used for robustness (no ECOS dependency issue).
# =============================================================

print("="*60)
print("Cell 34: PAPER TRADING — SIGNAL GENERATION")
print("="*60)

latest_date = model_df["date"].max()
print(f"Cell 34: Most recent date in dataset: {latest_date.date()}")

paper_train_start = latest_date - pd.DateOffset(years=2)
paper_train = model_df[
    (model_df["date"] >= paper_train_start) &
    (model_df["date"] < latest_date)
].dropna(subset=ALL_FEATURES + ["target"])

print(f"Cell 34: Training window: {paper_train['date'].min().date()} → {paper_train['date'].max().date()}")
print(f"Cell 34: Training rows: {len(paper_train):,}")

if len(paper_train) < 2000:
    print("Cell 34: WARNING — Insufficient training data.")
else:
    paper_model = make_pipeline(
        StandardScaler(),
        ElasticNet(alpha=best_en_params[0], l1_ratio=best_en_params[1], max_iter=5000)
    )
    paper_model.fit(paper_train[ALL_FEATURES], paper_train["target"])
    print("Cell 34: Model retrained on latest 2-year window.")

    paper_today = model_df[model_df["date"] == latest_date].dropna(subset=ALL_FEATURES).copy()
    print(f"Cell 34: Stocks with valid features on {latest_date.date()}: {len(paper_today)}")

    if len(paper_today) < 30:
        print("Cell 34: WARNING — Too few stocks with valid features today.")
    else:
        paper_today["pred"] = paper_model.predict(paper_today[ALL_FEATURES])

        n_pt = len(paper_today)
        paper_today = paper_today.sort_values("sec_id").reset_index(drop=True)
        # v17 (change 0.5): continuous forecasts instead of ranks —
        # same transform as the backtest optimizer (Cell 30)
        _p = paper_today["pred"]
        _z = ((_p - _p.mean()) / (_p.std() + 1e-12)).clip(-3.0, 3.0)
        mu_pt = _z / 3.0 * 0.5

        recent_rets = model_df[
            (model_df["date"] >= latest_date - pd.DateOffset(days=30)) &
            (model_df["sec_id"].isin(paper_today["sec_id"]))
        ].pivot(index="date", columns="sec_id", values="ret").fillna(0)

        cov_pt = recent_rets.reindex(columns=paper_today["sec_id"]).cov().values
        cov_pt = (cov_pt + cov_pt.T) / 2 + 1e-5 * np.eye(n_pt)

        w_pt = cp.Variable(n_pt)
        prob_pt = cp.Problem(
            cp.Maximize(mu_pt.values @ w_pt - RISK_AVERSION * cp.quad_form(w_pt, cov_pt)),
            [cp.sum(w_pt) == 0, cp.norm(w_pt, 1) <= 1.0,
             w_pt <= MAX_WEIGHT, w_pt >= -MAX_WEIGHT]
        )
        # Use SCS for robustness (no ECOS install required)
        prob_pt.solve(solver=cp.SCS, verbose=False)

        if w_pt.value is not None:
            paper_today["target_weight"] = w_pt.value
            paper_today["direction"] = paper_today["target_weight"].apply(
                lambda x: "LONG" if x > 0.001 else ("SHORT" if x < -0.001 else "FLAT")
            )
            orders = paper_today[paper_today["direction"] != "FLAT"][
                ["ticker","target_weight","direction","pred"]
            ].sort_values("target_weight", ascending=False)

            print(f"\nCell 30: Paper Trading Orders for {latest_date.date()}:")
            print(f"Cell 34:   Long positions:  {(orders['direction']=='LONG').sum()}")
            print(f"Cell 34:   Short positions: {(orders['direction']=='SHORT').sum()}")
            print("\nCell 30: Top 10 Long positions:")
            print(orders[orders["direction"]=="LONG"].head(10).to_string(index=False))
            print("\nCell 30: Top 10 Short positions:")
            print(orders[orders["direction"]=="SHORT"].tail(10).to_string(index=False))
        else:
            print("Cell 34: Optimisation failed for paper trading date.")

## Cell 35 — Assumptions, Limitations & Next Steps

### What Changed from v15 → v16

| Change | Impact |
|---|---|
| Fixed residualization (Cell 10) | v15's residualization cell was commented out entirely and never ran; it also used an invalid cross-sectional-per-date approach (the market factor has zero variance within a single day, so it cannot be regressed against that way). v16 restores the working per-ticker (time-series) version from v12, confirmed via a post-residualization correlation check landing at ~0.00000 |
| Multi-model walk-forward (Cell 22) | All 5 models are now walk-forward tested out-of-sample, not just whichever model looked best on the single static validation period — removes a subtle look-ahead in how the "best" model was chosen |
| Dynamic forecast combination (Cell 23) | v15 documented but never implemented trailing-IC ensemble weighting (the actual code was a stub aliasing the ensemble to one model). v16 implements it: weights are recomputed at every out-of-sample date from only the trailing 126 days, so the ensemble adapts as relative model performance changes, with zero use of future information |
| Coefficient stability chart fixed (Cell 25) | v15 broke this by switching its walk-forward to Random Forest-only (no `.coef_` attribute) without adapting the chart, causing a `KeyError` crash. v16's multi-model walk-forward keeps fitting ElasticNet alongside the other models specifically so coefficient tracking keeps working |

### What Changed from v11 → v12 (carried forward)

| Change | Impact |
|---|---|
| Residualization against market factor (Cell 10) | Stock features (momentum, vol, ma_spread) isolate stock-specific signal from broad market movement, addressing the concern that Random Forest importance was dominated by spy_tlt/hyg_tlt regime signals |
| Rolling IC chart (Cell 24) | Visualizes the dynamic ensemble's signal stability over time instead of only reporting a single summary statistic |
| Coefficient stability chart (Cell 25) | Directly addresses the assignment requirement that "coefficients should be relatively consistent across time" with a concrete visual and a sign-consistency metric |
| Plotted feature importance (Cell 19) | Random Forest importances have a proper bar chart in addition to the text-based view |

### What Changed from v10 → v11 (carried forward)

| Change | Impact |
|---|---|
| Point-in-time S&P 500 constituents | Eliminates survivorship bias — results are more conservative and realistic |
| Short volume features (3 signals) | Adds daily short-selling flow as a bearish signal |
| Short interest features (computed, but excluded from model) | Squeeze risk signal explored but not included — see below |
| Fundamental features (4 signals) | Adds quality and growth factors (ROE, margin, revenue growth, D/E) |
| 19 features total vs 12 in v10 | Richer signal set, but more opportunity for overfitting |

### Assumptions Made

| Assumption | Justification | Limitation |
|---|---|---|
| Point-in-time constituents from history file | Eliminates survivorship bias directly | History file records change dates only; days between changes use forward-fill |
| Short interest excluded from `ALL_FEATURES` entirely | Only 180/570+ tickers covered, starting Dec 2022 — including it in the global feature dropna previously collapsed the entire training set to ~19 days. Excluding it preserves the full multi-year training window for all other features | Short interest's potential predictive value (squeeze risk) is not captured by the model. Feature-level IC audit (Cell 16) on the brief overlap period showed negative IC, suggesting limited value even if coverage were fuller |
| Residualize stock features against spy_tlt only (per ticker, not sector or size) | Market beta is the single largest source of unwanted correlation in our feature set; per-ticker time-series regression is the mathematically valid way to estimate and remove it | A full Fama-French residualization (size, value, sector) would more completely isolate idiosyncratic signal; this is a simplified one-factor version |
| Multi-model walk-forward fits all 5 models every window | Avoids selecting "the best model" using validation data and then only testing that model out-of-sample, which double-uses the same information | 5x the per-window computation cost vs a single-model walk-forward; Random Forest uses reduced `n_estimators=50` (vs 100 in the static ladder) specifically to keep this tractable |
| Dynamic ensemble uses a 126-day (~6 month) trailing lookback for IC weighting | Long enough to estimate ICIR with reasonable stability, short enough to actually adapt to changing model performance | The lookback length itself is a parameter choice not tuned via cross-validation; a shorter or longer window could plausibly perform differently |
| Fundamentals lagged 1 day | Conservative safety margin | True lag is quarterly; full rigor needs earnings release dates |
| 10 bps execution cost | Realistic for S&P 500 institutional trading | Excludes short borrow cost and nonlinear market impact |
| FRED macro lagged 1 day | Prevents lookahead | CPI published ~2 weeks after month-end; 1-day lag is insufficient for CPI |
| 67/637 historical tickers (10.5%) missing price data | Yahoo Finance has no data for delisted/acquired/renamed tickers (e.g. TWTR, ANTM, ATVI) | Reintroduces a small residual survivorship-style gap for these specific names, despite the point-in-time constituent filter being correctly applied |

### Honest Limitations
- Short interest only covers **180 symbols** from Dec 2022 onwards — too sparse to include in the model without destroying earlier training data, so it is computed but excluded from `ALL_FEATURES`
- Fundamental data covers **633 symbols** — good but not complete coverage
- The historical constituent file records **change dates only** (~96 rows from 2020) — days between changes are forward-filled, which is correct but means very frequent intra-period additions/removals may have a small delay
- Survivorship bias is reduced but **not eliminated** — the history file itself may have gaps, and 67 tickers could not be downloaded at all
- Residualization is single-factor (market only) — does not control for sector or size effects
- The multi-model walk-forward is meaningfully slower than v12's single-model version, since it fits 5 models per window instead of 1 — expect noticeably longer runtime on Cell 22
- The dynamic ensemble's 126-day lookback, like any lookback length choice, is a judgment call rather than something formally optimized

### Next Steps
1. **Earnings release dates** — Use FRED vintage API or Compustat release dates to properly lag fundamental data
2. **Sector neutrality** — Add sector constraints to CVXPY to prevent unintended sector bets
3. **Multi-factor residualization** — Extend Cell 10 to also residualize against size and sector factors, not just market beta
4. **Expand short interest coverage** — Source data covering all 500+ S&P 500 stocks would make it viable to include without the dropna tradeoff; could also try a separate "short interest available" indicator + zero-fill instead of dropna-exclusion
5. **Level 6 portfolio** — Add square-root market impact model to CVXPY objective
6. **Live paper trading scheduler** — Automate Cell 32 to run daily and log results vs actual market moves
7. **Ticker rename mapping** — Build a lookup table for tickers that changed symbols (e.g. ANTM→ELV, DISCA→WBD) to recover some of the 67 currently-missing historical names
8. **Tune the dynamic ensemble's lookback window** — Test 63-day (~3 month) and 252-day (~1 year) lookbacks against the current 126-day choice to see how sensitive the dynamic weights are to this parameter
9. **Track Random Forest feature importance across walk-forward windows** — Cell 25 currently only tracks ElasticNet coefficients (the only model with a clean linear `.coef_`); an analogous stability check for RF's `feature_importances_` across windows would extend the same "is this signal durable or noise" question to the non-linear model


---

### What Changed from v16 → v17

| Change | Impact |
|---|---|
| CIK entity key (Cell 2b, used throughout) | Renamed companies (FB→META, ANTM→ELV, …) are one continuous entity via `sec_id = CIK:canonical_ticker`; joins to short-vol / fundamentals survive renames. Pure CIK alone can't be the row key because multi-class listings (GOOG/GOOGL, BRK-A/BRK-B) share one CIK — the ticker suffix keeps classes distinct. Caveat: price series across a rename boundary may carry a level artifact in long-horizon features (mom_252) |
| Turnover constraint in the MVO (Cell 30) | `‖w − w_prev‖₁ ≤ 1.50` per day (configurable). v16 had no turnover control; the optimizer now trades a persistent book instead of rebuilding it daily |
| Continuous µ in the optimizer (Cells 30, 34) | The optimizer sizes by z-scored continuous forecasts (clipped ±3), not just rank order. `USE_CONTINUOUS_MU=False` restores v16 behaviour |
| σ/√ADV liquidity term (Cells 5, 11, 30, 31) | Per-name Almgren-style linear impact coefficient λᵢ ∝ σᵢ/√(dollar ADVᵢ), normalised so the median name costs 10 bps per unit traded; enters the optimizer objective and the cost analysis. Untransformed `tc_sigma` / `tc_adv` copies are carried through Cell 11 so they escape the feature z-scoring |
| Sharpe by year (Cell 32) | Per-calendar-year Sharpe for each portfolio level — shows whether performance is concentrated in one regime |
| Hurst exponent factor (Cell 5b) | Rolling 126-day aggregated-variance Hurst estimator, recomputed weekly and ffilled for speed; distinguishes trending (H>0.5) from mean-reverting (H<0.5) names |
| Yang-Zhang volatility factor (Cell 5) | 20-day range-based OHLC vol — more efficient than close-to-close vol at the same window; added to the feature set alongside vol_20/vol_60 |
| Optional fundamental-factor residualization (Cells 12b, 13) | `RESIDUALIZE_VS_FUNDAMENTALS=True` cross-sectionally neutralises all stock features against ROE / margin / revenue growth / D-to-E per date, and drops fundamentals from `ALL_FEATURES` — a feature set containing no fundamental data, with fundamentals acting purely as risk factors. Off by default |

### v17 known limitations / notes
- The SEC ticker→CIK file maps **current** tickers only; old symbols rely on the manual `TICKER_RENAME_MAP` (extend it as more renames are found). Unmapped tickers fall back to a synthetic `NOCIK:` id — they still work, just without rename stability.
- Names dropping out of the tradable universe are treated as closed at zero cost in the optimizer's `w_prev` book (simplification).
- The Hurst estimator is recomputed every 5 days and forward-filled — a runtime/precision trade-off.
- λ = σ/√ADV is a **relative** liquidity scaling calibrated so the median name costs the base bps; it is not a NAV-aware participation model (a square-root impact model with an explicit portfolio size would be the Level 6 upgrade).


---

### What Changed from v17 → v18 (final)

| Change | Why |
|---|---|
| Target = per-date **demeaned 5-day forward return** (Cell 11, `TARGET_HORIZON` in Cell 1) | Removes the common market component from the target, so models can only score by *ranking stocks*, not timing the market; 5-day horizon chosen a priori on turnover economics (v17's 1-day horizon implied ~114x annual turnover, unsurvivable at realistic costs) |
| **Macro/ETF features removed from `ALL_FEATURES`** (Cell 13) | Identical for every stock within a day ⇒ zero cross-sectional ranking information by construction; their presence turned the RF into a market timer with constant within-day predictions (v17's "IC=+0.12, N=1" artifact). Retained only as the residualization factor and regime context |
| **Dynamic / IC-weighted combinations removed** (Cells 23, 25) | Trailing-ICIR weighting had no economic hypothesis — it selected whatever worked recently. Traded signal is one pre-registered model: walk-forward ElasticNet. 1/N linear average kept as a no-fitted-weights robustness check; RF kept as a non-traded non-linearity diagnostic |
| **Explicit feature hypotheses with expected signs** (intro table + Cell 16 check) | Every feature now has a stated directional hypothesis, checked against realised training IC — the audit reports agreement/disagreement rather than mining |
| **Horizon-aligned PnL** via `fwd_ret_1d` (Cells 11, 28, 30) | v17 credited weights formed at *t* with the *t−1→t* return; v18 portfolios earn the *t→t+1* return |
| **Solver failures carry the book forward** (Cell 30) | v17 dropped 147/834 days; failures cluster on stressed days, so dropping them was upward selection bias. v18 holds the previous book and reports carried-day counts |
| **Invalid-IC guard** (Cell 15) | IC summaries computed on < 30 valid days are flagged INVALID (v17's RF validation IC was one day) |
| **Rename map extended + canonical-symbol download fallback** (Cells 2b, 3) | Recovers price history for renamed names that failed in v17 (BLL→BALL, WLTW→WTW, CTL→LUMN, FLT→CPAY, PEAK→DOC, NLOK→GEN, CDAY→DAY, COG→CTRA, ADS→BFH, GPS→GAP). **Verify each mapping before publishing** — these were added from memory of corporate actions |
| Fundamental-residualization flag default **False** (Cell 12b) | Fundamentals enter as hypothesis-carrying features by default; the factor-neutralised variant remains one flag away |

### Interpretation guardrails for the write-up
- The **traded-signal OOS IC and its t-stat (Cell 25)** are the primary result. If |t| < 2, the honest headline is "no statistically significant cross-sectional signal at the weekly horizon on this universe/period" — that is a legitimate, defensible finding.
- Gross Level-5 Sharpe must be read **jointly with the IC**: a positive Sharpe with ~zero IC indicates construction effects (low-vol tilt, caps, turnover anchoring), not forecast skill. Attribute it as such.
- The net-of-cost number uses the σ/√ADV liquidity model with a 10 bps median name — report gross and net side by side.
- Remaining known gaps: ~10% of historical tickers still lack price data (acquired/delisted names with no successor symbol); borrow costs and nonlinear impact are not modelled; CPI/UNRATE use a 1-day lag rather than vintage release dates (macro now affects only residualization, not features).
- **Before publishing:** rotate or remove the FRED API key in Cell 1, and verify the entries added to `TICKER_RENAME_MAP`.


---

### v18-run audit → v19 corrections (each item: erroneous statement → fix)

| # | Erroneous statement / defect in the v18 run | v19 correction |
|---|---|---|
| 1 | **"Sharpe (annualised): 2.93 / 2.29 / 2.05 / 2.27"** — period returns are 5-day (≈50/yr) but `sharpe()` annualised with √252, inflating every portfolio Sharpe by √5 ≈ 2.24. True v18-run values ≈ 1.31 gross / 1.02 net / 0.92 / 1.01 | `PERIODS_PER_YEAR = 252/TARGET_HORIZON` (Cell 1); every portfolio-series `sharpe()` call now passes it (Cells 28, 29, 30, 31, 32) |
| 2 | **"Ann.Vol 0.22"** (Cell 32) — same error: `std × √252` on 5-day returns, overstated ×√5 (true ≈ 0.10) | `std × √PERIODS_PER_YEAR` |
| 3 | **"~112.7x annual portfolio turnover"** (Cell 31) — per-rebalance turnover ×252 instead of ×50.4; true ≈ 22.5x/yr | `× PERIODS_PER_YEAR` |
| 4 | **Lookahead (genuine): no purge at train/test boundaries.** With a 5-day forward target, the last 5 training dates of every walk-forward window — and of the static train and validation splits — had targets computed from returns *inside* the following window | Embargo: walk-forward trains on `unique_dates[i−TRAIN_WINDOW : i−TARGET_HORIZON]` (Cell 24); static split drops the last `TARGET_HORIZON` dates of train and of validation (Cell 14) |
| 5 | **"Volatility (daily)" / "Mean daily return" labels** on 5-day period statistics | Relabelled "per period" / "Mean period (5d) return" throughout |
| 6 | **Covariance lookback silently stretched**: after subsampling `unique_opt_dates[::5]`, `idx − 60` reached back 60 *rebalances* ≈ 300 trading days, not the intended 60 days | Lookback boundary computed on the daily calendar via `searchsorted` (Cell 30) |
| 7 | **24% solver-failure rate** (29/121 days carried) with ECOS→SCS | Clarabel→ECOS→SCS cascade (Cell 30); carry-forward retained as backstop |
| 8 | **Sharpe-by-year "—" for 2026** despite ~5 months of data — the ≥40-observation threshold assumed a daily series | Threshold now 20 rebalance periods |
| 9 | **Single arbitrary rebalance phase**: `dates[::5]` starting at the first date is 1 of 5 possible grids | Cell 28 reports Sharpe for all 5 phases and the range — report the range, not the best phase |
| 10 | **Avoidable survivorship from transient download failures** — the v18 run lost live large-caps (BK, MMC, K, CTRA, DAY, FI, …) to Yahoo flakiness, on top of genuinely delisted names | One individual retry pass per skipped ticker (and its canonical symbol) in Cell 3; remaining gaps are printed and should be listed as a limitation |

### Statistical honesty note for the write-up
After the √5 correction, the v18 run's numbers correspond to roughly **net Sharpe ≈ 1.0 over 121 rebalances (~2.4 years)** → t-stat ≈ 1.6, alongside a signal-IC t-stat of 1.75. Neither clears conventional significance. The v19 purge (item 4) removes a small upward bias, so expect the rerun to come in at or below these levels. Report the point estimate **with** its t-stat and the phase range — that is the defensible presentation.

In [ ]:
# =============================================================
# Headline result
# =============================================================
print("=" * 60)
print("CROSS-SECTIONAL EQUITY ALPHA - HEADLINE")
print("=" * 60)
print(f"Level 2 (gross notional)      Sharpe: {sharpe(daily_l2, PERIODS_PER_YEAR):8.4f}")
print(f"Level 4 (vol targeted)        Sharpe: {sharpe(daily_l4, PERIODS_PER_YEAR):8.4f}")
print(f"Level 5 MV (gross)            Sharpe: {sharpe(daily_l5, PERIODS_PER_YEAR):8.4f}")
print(f"Level 5 MV (net, liq-adj 10bps) Sharpe: {sharpe_net:8.4f}   <- headline")
print(f"  net max drawdown: {max_dd_net:.4f}   net total return: {equity_net.iloc[-1] - 1:.4f}")
print("=" * 60)
